In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import linear_model
from sklearn.model_selection import train_test_split

from datetime import datetime as dt

%matplotlib inline

path = "/Users/emiller/Documents/sf_tech/sf_datasets/"

parse_dates = ['aired_from', 'aired_to']
anime_csv = pd.read_csv(path + "anime.csv", parse_dates=parse_dates)

pd.set_option('display.max_columns', None)


print('complete')

complete


In [101]:
anime_df = anime_csv.drop(['url','title_japanese','title_synonyms','image_jpg_url', 'image_jpg_small_url',
       'image_jpg_large_url', 'image_webp_url', 'image_webp_small_url',
       'image_webp_large_url', 'trailer_youtube_id', 'trailer_url',
       'trailer_embed_url', 'trailer_image_url', 'trailer_small_image_url',
       'trailer_medium_image_url', 'trailer_large_image_url',
       'trailer_maximum_image_url','synopsis', 'background','aired_string','broadcast_string','licensors'], axis='columns')

anime_df.head()

,mal_id,approved,title,title_english,type,source,episodes,status,airing,duration,...,year,broadcast_day,broadcast_time,broadcast_timezone,producers,studios,genres,explicit_genres,themes,demographics
0,1,True,Cowboy Bebop,Cowboy Bebop,TV,Original,26.0,Finished Airing,False,24 min per ep,...,1998.0,Saturdays,01:00,Asia/Tokyo,"Bandai Visual, Victor Entertainment, Audio Pla...",Sunrise,"Action, Award Winning, Sci-Fi",NaN,"Adult Cast, Space",NaN
1,5,True,Cowboy Bebop: Tengoku no Tobira,Cowboy Bebop: The Movie,Movie,Original,1.0,Finished Airing,False,1 hr 55 min,...,NaN,NaN,NaN,NaN,"Sunrise, Bandai Visual",Bones,"Action, Sci-Fi",NaN,"Adult Cast, Space",NaN
2,6,True,Trigun,Trigun,TV,Manga,26.0,Finished Airing,False,24 min per ep,...,1998.0,Thursdays,01:15,Asia/Tokyo,Victor Entertainment,Madhouse,"Action, Adventure, Sci-Fi",NaN,Adult Cast,Shounen
3,7,True,Witch Hunter Robin,Witch Hunter Robin,TV,Original,26.0,Finished Airing,False,25 min per ep,...,2002.0,Wednesdays,01:25,Asia/Tokyo,"Bandai Visual, Dentsu, Victor Entertainment",Sunrise,"Action, Drama, Mystery, Supernatural",NaN,Detective,NaN
4,8,True,Bouken Ou Beet,Beet the Vandel Buster,TV,Manga,52.0,Finished Airing,False,23 min per ep,...,2004.0,Thursdays,18:30,Asia/Tokyo,"TV Tokyo, Dentsu",Toei Animation,"Action, Adventure, Fantasy",NaN,NaN,Shounen


In [102]:
# ............pick this back up here

"""
translate duration string into minutes integer
"""


def duration_minutes(duration):
    durations = duration.split()

    if 'min' in durations:
        minutes_val = int(durations[durations.index('min') - 1])
    else:
        minutes_val = 0

    if 'hr' in durations:
        hr_val = int(durations[durations.index('hr') - 1]) * 60
    else:
        hr_val = 0
    
    if 'sec' in durations:
        sec_to_minutes = 1
    else:
        sec_to_minutes = 0
    
    return (minutes_val + hr_val + sec_to_minutes)

anime_df['duration_minutes'] = anime_df['duration'].apply(duration_minutes)
anime_df.head()


,mal_id,approved,title,title_english,type,source,episodes,status,airing,duration,...,broadcast_day,broadcast_time,broadcast_timezone,producers,studios,genres,explicit_genres,themes,demographics,duration_minutes
0,1,True,Cowboy Bebop,Cowboy Bebop,TV,Original,26.0,Finished Airing,False,24 min per ep,...,Saturdays,01:00,Asia/Tokyo,"Bandai Visual, Victor Entertainment, Audio Pla...",Sunrise,"Action, Award Winning, Sci-Fi",NaN,"Adult Cast, Space",NaN,24
1,5,True,Cowboy Bebop: Tengoku no Tobira,Cowboy Bebop: The Movie,Movie,Original,1.0,Finished Airing,False,1 hr 55 min,...,NaN,NaN,NaN,"Sunrise, Bandai Visual",Bones,"Action, Sci-Fi",NaN,"Adult Cast, Space",NaN,115
2,6,True,Trigun,Trigun,TV,Manga,26.0,Finished Airing,False,24 min per ep,...,Thursdays,01:15,Asia/Tokyo,Victor Entertainment,Madhouse,"Action, Adventure, Sci-Fi",NaN,Adult Cast,Shounen,24
3,7,True,Witch Hunter Robin,Witch Hunter Robin,TV,Original,26.0,Finished Airing,False,25 min per ep,...,Wednesdays,01:25,Asia/Tokyo,"Bandai Visual, Dentsu, Victor Entertainment",Sunrise,"Action, Drama, Mystery, Supernatural",NaN,Detective,NaN,25
4,8,True,Bouken Ou Beet,Beet the Vandel Buster,TV,Manga,52.0,Finished Airing,False,23 min per ep,...,Thursdays,18:30,Asia/Tokyo,"TV Tokyo, Dentsu",Toei Animation,"Action, Adventure, Fantasy",NaN,NaN,Shounen,23


In [104]:


# fill Nan values for aired_prop_to_year with current year if airing is True
anime_df['aired_prop_to_year'] = anime_df['aired_prop_to_year'].where(anime_df['airing'] == False, anime_df['aired_prop_to_year'].fillna(dt.today().year))

anime_df['years_aired'] = anime_df['aired_prop_to_year'] - anime_df['aired_prop_from_year'] + 1
anime_df[['airing','aired_prop_to_year','aired_prop_from_year','years_aired']].head(20)


,airing,aired_prop_to_year,aired_prop_from_year,years_aired
0,False,1999.0,1998.0,2.0
1,False,NaN,2001.0,NaN
2,False,1998.0,1998.0,1.0
3,False,2002.0,2002.0,1.0
4,False,2005.0,2004.0,2.0
5,False,2008.0,2005.0,4.0
6,False,2005.0,2005.0,1.0
7,False,2003.0,2002.0,2.0
8,False,2006.0,2004.0,3.0
9,False,2005.0,2004.0,2.0


In [106]:
anime_df.columns

Index(['mal_id', 'approved', 'title', 'title_english', 'type', 'source',
       'episodes', 'status', 'airing', 'duration', 'rating', 'score',
       'scored_by', 'rank', 'popularity', 'members', 'favorites', 'aired_from',
       'aired_to', 'aired_prop_from_day', 'aired_prop_from_month',
       'aired_prop_from_year', 'aired_prop_to_day', 'aired_prop_to_month',
       'aired_prop_to_year', 'season', 'year', 'broadcast_day',
       'broadcast_time', 'broadcast_timezone', 'producers', 'studios',
       'genres', 'explicit_genres', 'themes', 'demographics',
       'duration_minutes', 'years_aired'],
      dtype='object')

In [ ]:
"""
GENRES:
for first iteration, remove genres field
for future iterations, create ~ 10 fields to capture top genres
"""

anime_df['type'] = anime_df['type'].str.lower()
anime_df['source'] = anime_df['source'].str.lower()

anime_df['content_rating'] = (
    anime_df['rating']
    .str.split().str[0]
    .str.lower()
    .str.replace('-','_', regex=False)
    .str.replace('+','_plus', regex=False)
    )

drop_list = ['title','title_english','status','duration','rating','aired_from','aired_to','seanson','broadcast_day'
             'broadcast_timezone','producers','studios','genres','explicit_genres','themes','demographics']



In [ ]:
# anime_type_dummies.head()
"""
anime_source_dummies.head()

need to...clean type and source fields:
lower case 
underscores
rename music --> 'music_type'
"""

# anime_df['type_c'] = anime_df['type'].str.lower().str.replace(' ', '_', regex=False)

# anime_df['type'].value_counts()

.......................start here
anime_df['source_c'] = anime_df['source'].str.lower().str.replace(' ', '_', regex=False).str.replace(' ', '_', regex=False)(columns={' ':'_','-':'_','light_novel':'novel', 'visual_novel':'novel', 'web_manga':'manga', '4_koma_manga':'manga', 'web_novel':'novel'})
anime_df.souce_c.value_counts()



TypeError: Series.rename() got an unexpected keyword argument 'columns'

In [ ]:
# get dummies
anime_type_dummies = pd.get_dummies(anime_df.type)
anime_type_dummies.rename(columns={'Music':'type_music'}, inplace=True)

anime_source_dummies = pd.get_dummies(anime_df.source)
anime_rating_dummies = pd.get_dummies(anime_df.rating)

# join dataframes with get dummies
anime_temp = anime_df.join(anime_type_dummies)
anime_temp2 = anime_temp.join(anime_source_dummies)
anime = anime_temp2.join(anime_rating_dummies)

anime.head()

ValueError: columns overlap but no suffix specified: Index(['Music'], dtype='object')

In [ ]:
'type' - format: TV/movie 
'source' - original vs manga
'episodes' - total number of episodes
'duration_minutes'
'rating' - content age rating 
'scored_by' - number of users that gave a score
'popularity' - rank based on members who have anime on list 
'members' - number of members with the anime on list (used for popularity?)
# 'aired_from' and 'aired_to' - total amount of time anime was aired 
'aired_prop_from_year' and 'aired_prop_to_year' - get number of years anime was aired 
'season' - season of year of original broadcast
'broadcast_day' - day of week anime was broadcast in japan (often null)
'broadcast_time' - time slot (in japan) that anime aired 
'studios' - studios responsible 
'genres'

'rank' - rank on list based on weighted score 
'score' - user score from 1 to 10


next...
x get number of years that anime was active


x simplify rating column (PG - Children --> PG...get distinct ratings and work from there...not needed)
x check broadcast_day for null counts - TOO MANY NULLS TO USE COLUMN
x look at format of times for broadcast_time - are AM and PM separated? yes, they use 24HR time
x are there 10 top studios? there are many studios...maybe have a categorical model later for this


are there 10 top genres?:
genre_dict = {'Comedy':'comedy', 'Fantasy':'fantasy', 'Hentai':'hentai', 'Slice of Life':'slice_of_life', 'Avant Garde':'avant_garde',
 'Action':'action', 'Adventure':'adventure', 'Drama':'drama','Horror':'horror','Mystery':'mystery','Sci-Fi':'sci_fi',
 'Romance':'romance','Suspense':'suspence', 'Supernatural':'supernatural'}


create one of these columns for each genre 
anime_df['g_comedy'] = anime_df['genres'].apply(lambda x: 'Comedy' in x.split(',') if isinstance(x, str) else False

check for collinearity - suspects:
episodes and broadcast_years
members and popularity
members and scored_by
type and duration_minutes?


target variables:
'score' (is rank the same...high collinearity)
'episodes'
